# Import Libraries

In [ ]:
# standard library
import os
import re
import sys
import subprocess
import importlib.util
import random

# third-party
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.regressionplots import plot_partregress
import joblib

# local imports
from scripts.helpers import map_to_color
from scripts.lower_dim import compute_pcs

from helper import *
from helper.analysis import build_effects_from_correlated_snps, gwas_linregress, summarize_component_fits, center_geno_by_ps, get_n_pcs
from helper.poly_analysis import polygenic_noise

In [ ]:
def fit_line(x_, y_):
    slope, intercept, r, p, se = stats.linregress(x_, y_)
    neglogp = -np.log10(p) if p > 0 else np.inf
    return slope, intercept, neglogp

# Load data

In [ ]:
raw_geno, humans = load_data()

In [ ]:
pheno = make_pheno(humans)

In [ ]:
show_biases(pheno)

In [ ]:
chosen_bias = "discrete"
pheno[chosen_bias] = (pheno[chosen_bias] - pheno[chosen_bias].mean()) / pheno[chosen_bias].std()

In [ ]:
G, snp_cols, pcs = _prep_geno_and_pcs(raw_geno, PCs=5)
G.shape, len(snp_cols), pcs.shapew

In [ ]:
values, pcs, ps, HWE_dev = find_snps(raw_geno, pop=humans["populations"], chosen_bias=pheno["discrete"], PCs=5)

In [ ]:
visualize_grid_and_pcs(pcs, humans, dpi=36, s=48)

In [ ]:
show_top_snps_ordered(humans, raw_geno, values)

In [ ]:
chosen_snp = values.loc[
    values["reasons"] == "lowest pval for bias (PC-corrected)", "names"
].iloc[0]

# Standardize genotype

In [ ]:
geno, std_params = standardize_and_return_params(raw_geno)

In [ ]:
# Find the snps most and least correlated with chosen snp

In [ ]:
mean_abs_corr, corr_df_sorted, correlated_snps = snp_correlation_analysis(
    geno,chosen_snp,
    top_k=5
)

In [ ]:
show_corr_snps_ordered(humans, geno, correlated_snps)

# generate Polygenic Noise

In [ ]:
chosen_snp = chosen_snp
effect_chosen_snp = 1
chosen_bias = chosen_bias
effect_bias = 1
qty_neg_correlated_with_chosen_snp = 6
effect_neg_corrs = 1
effect_neg_corrs = effect_neg_corrs/qty_neg_correlated_with_chosen_snp
qty_pos_correlated_with_chosen_snp = 6
effect_pos_corrs = 1
effect_pos_corrs = effect_pos_corrs/qty_pos_correlated_with_chosen_snp
percentage_of_polygenic_snps = 20
effect_poly_snps = 1

# effect for the chosen causal snp
pheno['chosen_snp_effect'] = geno[chosen_snp]*effect_chosen_snp
# effect for the most correlared snps
neg_corr_snps = list(corr_df_sorted.tail(qty_neg_correlated_with_chosen_snp)['snp'])
pos_corr_snps = list(corr_df_sorted.head(qty_pos_correlated_with_chosen_snp)['snp'])

pheno['neg_corr_snps_effect'] = 0
pheno['pos_corr_snps_effect'] = 0
for snp in neg_corr_snps:
    pheno['neg_corr_snps_effect'] += geno[snp]*effect_neg_corrs
for snp in pos_corr_snps:
    pheno['pos_corr_snps_effect'] += geno[snp]*effect_pos_corrs

nr_of_snps_to_select = int((geno.shape[1]*percentage_of_polygenic_snps)/100)
excluded_snps = set([chosen_snp]) | set(neg_corr_snps) | set(pos_corr_snps)
candidate_snps = [snp for snp in geno.columns if snp not in excluded_snps]
if nr_of_snps_to_select > len(candidate_snps):
    raise ValueError("Not enough candidate SNPs to sample from.")
selected_poly_snps = random.sample(candidate_snps, nr_of_snps_to_select)

pheno['polygenic_effect'] = 0
per_snp_effect = effect_poly_snps / nr_of_snps_to_select

for snp in selected_poly_snps:
    pheno['polygenic_effect'] += geno[snp] * per_snp_effect

pheno['total_pre_pop'] = pheno["chosen_snp_effect"] + pheno["pos_corr_snps_effect"] + pheno["neg_corr_snps_effect"] + pheno["polygenic_effect"]
pheno['total'] = pheno[chosen_bias] + pheno['total_pre_pop']


snps_betas = pd.concat(
    [
        pd.DataFrame({
            "snp": [chosen_snp],
            "beta": [effect_chosen_snp],
            "category": ["chosen"],
        }),
        pd.DataFrame({
            "snp": neg_corr_snps,
            "beta": effect_neg_corrs,
            "category": "neg_corr",
        }),
        pd.DataFrame({
            "snp": pos_corr_snps,
            "beta": effect_pos_corrs,
            "category": "pos_corr",
        }),
        pd.DataFrame({
            "snp": selected_poly_snps,
            "beta": per_snp_effect,  # effect_poly_snps / nr_of_snps_to_select
            "category": "polygenic",
        }),
    ],
    ignore_index=True,
)

# --- enforce uniqueness BEFORE merge (critical) ---
snps_betas = snps_betas.drop_duplicates(subset="snp", keep="first")

# --- expand to all SNPs, fill missing with zero, enforce order ---
snps_betas = (
    pd.DataFrame({"snp": geno.columns})
    .merge(snps_betas, on="snp", how="left")
    .assign(
        beta=lambda df: df["beta"].fillna(0),
        category=lambda df: df["category"].fillna("null"),
    )
    .set_index("snp")
    .loc[geno.columns]
    .reset_index()
)


In [ ]:
snps_betas

In [ ]:
cols = [chosen_bias, "chosen_snp_effect", "pos_corr_snps_effect", "neg_corr_snps_effect","polygenic_effect","total_pre_pop","total"]
plot_effects(pheno,cols)

# Vizualize

In [ ]:
pheno['snp'] = pheno['chosen_snp_effect']
pheno['pop'] = pheno[chosen_bias]
pheno['poly'] = pheno["pos_corr_snps_effect"] + pheno["neg_corr_snps_effect"] + pheno["polygenic_effect"]

In [ ]:
params = plot_components_vs_snp(geno, chosen_snp, pheno)

# perceived betas pre population bias

In [ ]:
perceived_beta = []
neglops = []
for snp in geno.columns:
    slope, intercept, neglogp = fit_line(geno[snp], pheno['total_pre_pop'])
    perceived_beta.append(slope)
    neglops.append(neglogp)

snps_betas['perceived_beta_pre_pop'] = perceived_beta
snps_betas['neglogp_pre_pop'] = neglops


perceived_beta = []
neglops = []
for snp in geno.columns:
    slope, intercept, neglogp = fit_line(geno[snp], pheno['total'])
    perceived_beta.append(slope)
    neglops.append(neglogp)

snps_betas['perceived_beta_total'] = perceived_beta
snps_betas['neglogp_total'] = neglops
snps_betas['marginal_effect'] = snps_betas["beta"] - snps_betas['perceived_beta_pre_pop']

In [ ]:
# Estimated marginal effect pre correction

In [ ]:
snps_pcs = get_n_pcs(geno.T, 25)

In [ ]:
def estimate_marginal_effect(pc_df, effect_df, effect, chosen_snp):
    temp_df = pd.DataFrame()
    chosen_vec = pc_df.loc[chosen_snp].values
    all_vecs = pc_df.values
    distances = np.linalg.norm(all_vecs - chosen_vec, axis=1)
    temp_df['names'] = pc_df.index
    temp_df['distances'] = distances
    temp_df = temp_df[temp_df['names']!=chosen_snp]
    merged = temp_df.merge(effect_df,left_on="names",right_on="index",how="inner")

    # compute weighted sum
    weighted_sum = (merged["distances"] * merged[effect]).sum()/len(pc_df)
    return weighted_sum

In [ ]:
esti_marg_effects=[]
for snp in snps_betas["index"]:
    esti_marg_effects.append(estimate_marginal_effect(snps_pcs, snps_betas, "perceived_beta_total", snp))

In [ ]:
snps_betas['esti_marginal_pre_corr'] = esti_marg_effects

In [ ]:
sns.set(style="whitegrid")

plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=snps_betas,
    x="esti_marginal_pre_corr",
    y="marginal_effect",
    hue="beta",
    alpha=0.6,
    s=30
)

plt.axhline(0, color="grey", linestyle="--", linewidth=1)
plt.axvline(0, color="grey", linestyle="--", linewidth=1)

plt.xlabel("Beta marginal effect")
plt.ylabel("Estimated perceived beta (with pop)")
plt.title("Marginal vs Estimated Perceived Effects")

plt.tight_layout()
plt.show()


# With correction

In [ ]:
pheno['null'] = 0

In [ ]:
geno_corr = raw_geno - ((ps - (1-ps))+1)/2

In [ ]:
category_to_effect = {
    "null": "null",
    "polygenic": "polygenic_effect",
    "neg_corr": "neg_corr_snps_effect",
    "pos_corr": "pos_corr_snps_effect",
    "chosen": "chosen_snp_effect",
}


In [ ]:
snp_cors = []
beta_cors = []
for snp in snps_betas.index:
    category = snps_betas.loc[snp,"category"]
    snp_name = snps_betas.loc[snp,"index"]
    
    effect = category_to_effect.get(category)
    slope, intercept, neglogp = fit_line(geno_corr[snp_name],pheno[effect])
    snp_cors.append(snp_name)
    beta_cors.append(slope)

In [ ]:
snp_beta_corr = pd.DataFrame({
    "index": snp_cors,
    "beta": beta_cors,
})

In [ ]:
geno_corr = raw_geno - ((ps - (1-ps))+1)/2

In [ ]:
std_corr = (((ps*(1-ps)*2)+1)/2).mean(axis=0)

In [ ]:
geno_corr = geno_corr/std_corr

In [ ]:
perceived_beta = []
neglops = []
for snp in geno.columns:
    slope, intercept, neglogp = fit_line(geno_corr[snp], pheno['total_pre_pop'])
    perceived_beta.append(slope)
    neglops.append(neglogp)

snp_beta_corr['perceived_beta_pre_pop_corr'] = perceived_beta
snp_beta_corr['neglogp_pre_pop_corr'] = neglops
 

perceived_beta = []
neglops = []
for snp in geno.columns:
    slope, intercept, neglogp = fit_line(geno_corr[snp], pheno['total'])
    perceived_beta.append(slope)
    neglops.append(neglogp)

snp_beta_corr['perceived_beta_total_corr'] = perceived_beta
snp_beta_corr['neglogp_total_corr'] = neglops

snp_beta_corr['marginal_effect_corr'] = snp_beta_corr["beta"] - snp_beta_corr['perceived_beta_pre_pop_corr']

In [ ]:
snps_pcs = get_n_pcs(geno_corr.T, 25)

In [ ]:
esti_marg_effects=[]
for snp in snps_betas["index"]:
    esti_marg_effects.append(estimate_marginal_effect(snps_pcs, snp_beta_corr, "perceived_beta_total_corr", snp))

In [ ]:
snp_beta_corr['esti_marginal_corr'] = esti_marg_effects

In [ ]:
sns.set(style="whitegrid")

plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=snp_beta_corr,
    x="esti_marginal_corr",
    y="marginal_effect_corr",
    hue="beta",
    alpha=0.6,
    s=30
)

plt.axhline(0, color="grey", linestyle="--", linewidth=1)
plt.axvline(0, color="grey", linestyle="--", linewidth=1)

plt.xlabel("Beta marginal effect")
plt.ylabel("Estimated perceived beta (with pop)")
plt.title("Marginal vs Estimated Perceived Effects")

plt.tight_layout()
plt.show()


In [ ]:
params = plot_components_vs_snp(geno, chosen_snp, effects)

In [ ]:
params

In [ ]:
corr = ((p-(1-p)+1)/2)
corr = (corr - snp_mean) / snp_std

In [ ]:
params_corr = plot_components_vs_snp(geno-corr, chosen_snp, effects)

In [ ]:
# For all other SNPs now

In [ ]:
effects["total_minus_snp_effect"] = effects["total"] - params_corr['total']['beta']*geno[chosen_snp]

In [ ]:
len(geno.columns.drop(chosen_snp))

In [ ]:
rows = []
to_use = geno-corr
for snp in geno.columns.drop(chosen_snp):
    slope, intercept, neglogp = fit_line(
        to_use[snp],
        effects["total_minus_snp_effect"]
    )
    rows.append({
        "snp": snp,
        "slope": slope,
        "intercept": intercept,
        "neglogp": neglogp
    })

vals = pd.DataFrame(rows)


In [ ]:
vals

In [ ]:
error_beta = params_corr['total']['beta']-params_corr['snp']['beta']
error_beta

In [ ]:
vals['slope'].mean()

In [ ]:
vals["slope"].sum()

In [ ]:
def plot_components_vs_x(x, y, covs=None, jitter=0.03):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from scipy import stats

    if covs is None:
        covs = []

    # coerce x
    x = np.asarray(x, dtype=float)

    # coerce y into a dict of named components
    if isinstance(y, dict):
        ys = {k: np.asarray(v, dtype=float) for k, v in y.items()}
    else:
        ys = {"total": np.asarray(y, dtype=float)}

    # coerce covariates (currently only for NaN-alignment)
    covs = [np.asarray(c, dtype=float) for c in covs]

    # build joint matrix for NaN filtering
    X = np.column_stack([x] + list(ys.values()) + covs)
    X = X[~np.isnan(X).any(axis=1)]

    x = X[:, 0]
    n_y = len(ys)
    y_keys = list(ys.keys())
    ys = {k: X[:, i + 1] for i, k in enumerate(y_keys)}
    covs = [X[:, 1 + n_y + i] for i in range(len(covs))]  # kept for future use

    def fit_line(x_, y_):
        slope, intercept, r, p, se = stats.linregress(x_, y_)
        neglogp = -np.log10(p) if p > 0 else np.inf
        return slope, intercept, neglogp

    def pretty(name, slope, intercept, neglogp):
        ptxt = "∞" if np.isinf(neglogp) else f"{neglogp:.2f}"
        return f"{name}: ŷ={intercept:.3f} {slope:+.3f}x, −log10(p)={ptxt}"

    # defaults similar to your original
    default_colors = {
        "snp":   "#c45508",
        "pop":   "#1f3a5f",
        "poly":  "#f2c14e",
        "total": "#000000",
    }

    order = [k for k in ["snp", "pop", "poly", "total"] if k in ys] + \
            [k for k in y_keys if k not in {"snp", "pop", "poly", "total"}]

    plt.figure(figsize=(7.5, 5.5))

    rng = np.random.default_rng(0)

    # scatter + mean±SEM only if x looks discrete-ish and "total" exists
    if "total" in ys:
        plt.scatter(
            x + rng.normal(0, jitter, size=len(x)),
            ys["total"],
            alpha=0.25,
            s=18,
            color="0.5",
        )

        uniq = np.unique(x)
        if uniq.size <= 20:  # heuristic: treat as grouped genotype-like
            for g in sorted(uniq):
                mask = (x == g)
                if mask.sum() >= 2:
                    mean = ys["total"][mask].mean()
                    sem = ys["total"][mask].std(ddof=1) / np.sqrt(mask.sum())
                else:
                    mean = ys["total"][mask].mean()
                    sem = 0.0

                plt.errorbar(
                    [g], [mean], yerr=[sem],
                    fmt="o", capsize=4,
                    color=default_colors.get("total", "black"),
                )

            mean_sem_handle = Line2D(
                [0], [0],
                marker="o",
                linestyle="none",
                markersize=7,
                markerfacecolor="none",
                markeredgewidth=1.5,
                color="black",
                label="mean ± SEM (total)",
            )
        else:
            mean_sem_handle = None
    else:
        mean_sem_handle = None

    x_line = np.array([x.min(), x.max()])

    params = {}
    for name in order:
        slope, intercept, neglogp = fit_line(x, ys[name])
        params[name] = {"beta": slope, "intercept": intercept, "neglogp": neglogp}

        color = default_colors.get(name, None)
        plt.plot(
            x_line,
            intercept + slope * x_line,
            linewidth=3 if name == "total" else 2.5,
            color=color,
            label=pretty(name, slope, intercept, neglogp),
            zorder=3 if name == "total" else 2,
        )

    # axis/labels
    uniq = np.unique(x)
    if uniq.size <= 20:
        plt.xticks(sorted(uniq))

    plt.xlabel("x")
    plt.ylabel("y / effect")
    plt.title("components vs x")

    # legend
    handles, labels = plt.gca().get_legend_handles_labels()
    if mean_sem_handle is not None:
        handles.append(mean_sem_handle)
        labels.append(mean_sem_handle.get_label())
    plt.legend(handles, labels, frameon=False, fontsize=8)

    plt.tight_layout()
    plt.show()

    return params


In [ ]:
#covs = [pcs[c] for c in [pc1, pc2]]
y = geno[chosen_snp]          # e.g. something = "height"


In [ ]:
plot_components_vs_x(geno-corr, y, covs=None, jitter=0.03)

In [ ]:
params_corr

In [ ]:
params_corr['total']['beta']

In [ ]:
Expected_snp_effect = params_corr['total']['beta']*geno[chosen_snp] + params['total']['intercept']

In [ ]:
Expected_snp_effect

In [ ]:
effects['total'].mean()